# Data Setup — Pneumonia Chest X-Ray Datasets

**AAI-540-02 Final Project · Group 4 · Standard Medical Models**

This notebook is the first step in the pipeline. It downloads the two public chest X-ray datasets used by the project and uploads them, organized by `source / split / label`, into the project's S3 data lake (`s3://pneumonia-data-set-group-4/raw-images/`).

**Datasets**
1. **Kermany Chest X-Ray (Pneumonia)** — pediatric JPEGs, ~5.8K images, pre-split into `train/val/test` × `NORMAL/PNEUMONIA`. Pulled programmatically via `kagglehub`.
2. **RSNA Pneumonia Detection Challenge** — adult DICOMs, ~30K images with a separate labels CSV. Pulled manually (see notes in Section 4) and split here into `train/val/test` using stratified-ish random sampling on the row index.

**Downstream steps:** preprocessing (`data_preperations*.ipynb` → `img_preprocessing.py`) reads from `raw-images/`, writes normalized PNGs to `preprocessed-images/`, and registers metadata in Athena (`pneumonia_db.image_metadata`).

## 1. Environment Setup

Install pinned `kagglehub` and import the libraries used across the notebook.
`boto3` and `sklearn` are imported in the sections where they're first used.

In [1]:
!pip install kagglehub -q

In [2]:
import os
from pathlib import Path

import pandas as pd
import kagglehub

from config import BUCKET_NAME, RAW_IMAGE_FOLDER

## 2. Download Kermany Chest X-Ray Dataset

Public Kaggle dataset: `paultimothymooney/chest-xray-pneumonia`. JPEGs, pre-split into `train/val/test` × `NORMAL/PNEUMONIA`. `kagglehub` downloads and caches the archive locally under `~/.cache/kagglehub/`.

In [3]:
# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Path to dataset files: /home/sagemaker-user/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2


In [4]:
# Wrap the cache path as a pathlib.Path for easy traversal.
chest_path = Path(path)

### 2.1 Locate the image root

The archive nests an extra `chest_xray/` directory. The actual `train/val/test` folders live two levels deep — bind the true root to `tru_chest_path` for the upload step.

In [5]:
[x for x in (chest_path / 'chest_xray'/ 'chest_xray').iterdir()]

[PosixPath('/home/sagemaker-user/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/chest_xray/.DS_Store'),
 PosixPath('/home/sagemaker-user/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/chest_xray/test'),
 PosixPath('/home/sagemaker-user/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/chest_xray/train'),
 PosixPath('/home/sagemaker-user/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/chest_xray/val')]

In [6]:
tru_chest_path = chest_path / 'chest_xray' / 'chest_xray'

## 3. Upload Kermany Images to S3

Walk every `.jpeg` under the local cache and upload it to the project bucket, preserving the `split/label/` directory structure. The destination prefix is `raw-images/` — the preprocessing pipeline reads from here.

The first cell below also **creates the S3 bucket if it doesn't exist yet** (idempotent — no-op on re-runs and for graders running the notebook against a pre-provisioned bucket).

**Note:** the RSNA uploads in Section 5 reuse the same `s3` client and bucket name.

In [7]:
import boto3
import botocore

s3 = boto3.client("s3")
bucket = BUCKET_NAME
root_folder = Path(RAW_IMAGE_FOLDER)


def ensure_bucket_exists(s3_client, bucket_name, region):
    """Create the bucket if it doesn't already exist.

    S3 bucket names are globally unique, so a successful head_bucket means we
    own (or at least can access) it; NoSuchBucket / 404 means we should create.
    Note: us-east-1 is the one region where CreateBucketConfiguration must be
    omitted, so we branch on that.
    """
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f"Bucket s3://{bucket_name} already exists.")
        return
    except botocore.exceptions.ClientError as e:
        code = e.response["Error"]["Code"]
        if code not in ("404", "NoSuchBucket"):
            raise

    if region == "us-east-1":
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region},
        )
    print(f"Created bucket s3://{bucket_name}.")


region = boto3.Session().region_name or "us-east-1"
ensure_bucket_exists(s3, bucket, region)

Bucket s3://pneumonia-data-set-am already exists.


In [ ]:
# First, count files so we can render a percentage.
# Note: rglob materializes lazily, but a list() pass over the local cache is cheap.
files = list(tru_chest_path.rglob('*.jpeg'))
total = len(files)
print(f'Uploading {total} JPEG files to s3://{bucket}/{root_folder}/ ...')

for i, file in enumerate(files, start=1):
    s3_dest = root_folder / file.relative_to(tru_chest_path)
    s3.upload_file(str(file), bucket, str(s3_dest))
    # \r overwrites the same line; end='' prevents a newline; flush=True forces an
    # immediate redraw (Jupyter buffers stdout otherwise).
    print(f'\r  [{i}/{total}] {i/total:6.1%}  {s3_dest}', end='', flush=True)

print()  # Final newline so the next cell's output starts on a fresh line.
print(f'Done. Uploaded {total} files.')

## 4. Download RSNA Pneumonia Detection Dataset

RSNA is hosted as a **Kaggle Competition** (`rsna-pneumonia-detection-challenge`), so it requires authenticated API access rather than the public-dataset endpoint used in Section 2. The download is ~3.7 GB of DICOMs and a labels CSV.

**Before you run this section:**
1. Create an API token at https://www.kaggle.com/settings ("Create New Token") — Kaggle returns a token string.
2. Visit https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules and click **"I Understand and Accept"** — Kaggle returns 401 / 403 on competition data until the rules are accepted by the authenticated account.
3. Install the token in your SageMaker (or local) home directory. In a terminal, replacing `API_TOKEN` with the value from step 1:
   ```bash
   mkdir -p ~/.kaggle && echo "API_TOKEN" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
   ```
   `kagglehub` discovers the token file automatically — no environment variables and no username needed.

`kagglehub` caches under `~/.cache/kagglehub/`, so re-running this cell after a successful download is a no-op.

In [8]:
# Download (or pick up from the local kagglehub cache).
# Requires that the authenticated account has accepted the RSNA competition rules.
rsna_path_str = kagglehub.competition_download('rsna-pneumonia-detection-challenge')
rsna_path = Path(rsna_path_str)

print(f"RSNA dataset path: {rsna_path}")

  0%|          | 0.00/3.66G [00:00<?, ?B/s]

  0%|          | 7.00M/3.66G [00:00<00:56, 69.5MB/s]

  1%|          | 24.0M/3.66G [00:00<00:29, 130MB/s] 

  1%|          | 40.0M/3.66G [00:00<00:26, 147MB/s]

  1%|▏         | 55.0M/3.66G [00:00<00:40, 95.0MB/s]

...

 99%|█████████▉| 3.64G/3.66G [00:54<00:00, 115MB/s]

100%|█████████▉| 3.66G/3.66G [00:54<00:00, 137MB/s]

100%|██████████| 3.66G/3.66G [00:54<00:00, 72.6MB/s]

Extracting files...


RSNA dataset path: /home/sagemaker-user/.cache/kagglehub/competitions/rsna-pneumonia-detection-challenge


In [9]:
# Sanity-check the archive contents. Section 5 expects these specific files to exist.
expected = [
    'stage_2_train_labels.csv',
    'stage_2_train_images',
    'stage_2_test_images',
]
missing = [name for name in expected if not (rsna_path / name).exists()]
if missing:
    raise FileNotFoundError(f"RSNA archive is missing expected files: {missing}")

print("RSNA archive contents:")
for p in sorted(rsna_path.iterdir()):
    print(f"  {p.name}")

RSNA archive contents:
  GCP Credits Request Link - RSNA.txt
  stage_2_detailed_class_info.csv
  stage_2_sample_submission.csv
  stage_2_test_images
  stage_2_train_images
  stage_2_train_labels.csv


## 5. Split RSNA and Upload to S3

RSNA arrives as a flat directory of `.dcm` files plus a single `stage_2_train_labels.csv` mapping `patientId → Target` (0 = normal, 1 = pneumonia). To match the Kermany layout we:

1. Random-split the row index into 80/10/10 train/val/test.
2. Map each row to its split folder and to a `NORMAL`/`PNEUMONIA` target folder.
3. Resolve each `patientId` to the on-disk `.dcm` path.
4. Upload to `s3://pneumonia-data-set-group-4/raw-images/<split>/<label>/<patientId>.dcm`.

_The downstream preprocessing notebook re-stratifies into the final 40/10/10/40 train/val/test/production split — this split is just to mirror the Kermany folder shape._

In [10]:
# Reuse the path produced by Section 4's kagglehub.competition_download call.
set_2_path = rsna_path

In [11]:
[p for p in set_2_path.iterdir()]

[PosixPath('/home/sagemaker-user/.cache/kagglehub/competitions/rsna-pneumonia-detection-challenge/GCP Credits Request Link - RSNA.txt'),
 PosixPath('/home/sagemaker-user/.cache/kagglehub/competitions/rsna-pneumonia-detection-challenge/stage_2_detailed_class_info.csv'),
 PosixPath('/home/sagemaker-user/.cache/kagglehub/competitions/rsna-pneumonia-detection-challenge/stage_2_sample_submission.csv'),
 PosixPath('/home/sagemaker-user/.cache/kagglehub/competitions/rsna-pneumonia-detection-challenge/stage_2_test_images'),
 PosixPath('/home/sagemaker-user/.cache/kagglehub/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images'),
 PosixPath('/home/sagemaker-user/.cache/kagglehub/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv')]

### 5.1 Load labels and assign train/val/test splits

In [12]:
# stage_2_train_labels.csv maps patientId -> {x, y, width, height, Target}
# (bounding boxes are NaN for negative cases).
df = pd.read_csv(set_2_path / 'stage_2_train_labels.csv')

In [13]:
df.head(2)

,patientId,x,y,width,height,Target
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,NaN,NaN,NaN,NaN,0
1,00313ee0-9eaa-42f4-b0ab-c148ed3241cd,NaN,NaN,NaN,NaN,0


In [14]:
from sklearn.model_selection import train_test_split

# 80/10/10 split on the row index. A fixed random_state would make this reproducible;
# left unseeded here because the canonical split is re-derived downstream in CNN_Model.ipynb.
index = list(df.index)
train_index, test_val_index = train_test_split(index, test_size=0.2)
test_index, val_index = train_test_split(test_val_index, test_size=0.5)

# Build a single index -> split mapping for vectorized assignment.
folder_map = {}
for i in train_index:
    folder_map[i] = 'train'
for i in val_index:
    folder_map[i] = 'val'
for i in test_index:
    folder_map[i] = 'test'

df['folder'] = df.index.map(folder_map)

In [15]:
df.folder.value_counts()

folder
train    24181
val       3023
test      3023
Name: count, dtype: int64

### 5.2 Resolve each `patientId` to its DICOM path and assign label folder

In [16]:
# Build patientId -> local .dcm path lookup from the on-disk archive.
img_path_map = {p.stem: p for p in set_2_path.rglob('*.dcm')}

df['local_path'] = df.patientId.map(img_path_map)

In [17]:
# Match the Kermany folder convention: 0 -> NORMAL, 1 -> PNEUMONIA.
target_map = {0: 'NORMAL', 1: 'PNEUMONIA'}
df['target_folder'] = df.Target.map(target_map)

In [18]:
df.head(2)

,patientId,x,y,width,height,Target,folder,local_path,target_folder
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,NaN,NaN,NaN,NaN,0,train,/home/sagemaker-user/.cache/kagglehub/competit...,NORMAL
1,00313ee0-9eaa-42f4-b0ab-c148ed3241cd,NaN,NaN,NaN,NaN,0,train,/home/sagemaker-user/.cache/kagglehub/competit...,NORMAL


In [19]:
len(df)

30227

### 5.3 Upload RSNA DICOMs to S3

Reuses the `s3` client and `bucket` defined in Section 3. Destination layout matches Kermany: `raw-images/<split>/<label>/<patientId>.dcm`.

In [20]:
rows = df.to_dict(orient='records')
total = len(rows)
print(f'Uploading {total} DICOM files to s3://{bucket}/{root_folder}/ ...')

# RSNA has ~30K files, so we throttle redraws to every 100 iterations
# (plus the final one) to keep the notebook output manageable.
REDRAW_EVERY = 100

for i, row in enumerate(rows, start=1):
    local_path = row['local_path']
    s3_dest = root_folder / row['folder'] / row['target_folder'] / local_path.name
    s3.upload_file(str(local_path), bucket, str(s3_dest))

    if i % REDRAW_EVERY == 0 or i == total:
        print(f'\r  [{i}/{total}] {i/total:6.1%}  {s3_dest}', end='', flush=True)

print()  # Final newline so the next cell's output starts cleanly.
print(f'Done. Uploaded {total} files.')

Uploading 30227 DICOM files to s3://pneumonia-data-set-am/raw-images/ ...


  [100/30227]   0.3%  raw-images/train/NORMAL/024cff3e-86d2-4658-b204-79b94b74b94f.dcm

  [200/30227]   0.7%  raw-images/train/NORMAL/05e79372-5e61-4957-bedf-c2c51a0757ad.dcm

  [300/30227]   1.0%  raw-images/train/PNEUMONIA/06b52ffd-71b8-429a-a284-bb39240ed343.dcm

  [400/30227]   1.3%  raw-images/train/NORMAL/074aea8f-d454-4590-9178-3183dbddcf30.dcm

...

  [30000/30227]  99.2%  raw-images/train/PNEUMONIA/2c9a388f-0042-4b88-b52b-ea0b21fb7960.dcm

  [30100/30227]  99.6%  raw-images/train/NORMAL/2faeab4f-17bf-410e-99b9-77bf4ae75963.dcm

  [30200/30227]  99.9%  raw-images/train/PNEUMONIA/c1415e26-fddf-4a0c-a7eb-7b9a0d9e9983.dcm

  [30227/30227] 100.0%  raw-images/train/PNEUMONIA/c1f7889a-9ea9-4acb-b64c-b737c929599a.dcm


Done. Uploaded 30227 files.


---

**Done.** Both datasets now live under `s3://pneumonia-data-set-group-4/raw-images/`, organized as `<source>/<split>/<label>/<file>`. Next step: `data_preperations.ipynb` for the preprocessing pipeline.